# Photos Library Duplicate Cleanup Notebook

Report-only v1. This notebook does **not** delete anything.

Run order:
1. Run configuration.
2. Load/build inventory.
3. Fill identity fields.
4. Group and analyze duplicate candidates.
5. Write permanent operation report.

Helper functions live in `photos_duplicate_cleanup_helpers.py` so the notebook stays readable.


In [1]:
from pathlib import Path
from datetime import datetime
import re
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from explorephotoslibrary import *

CACHE_DIR = Path("data/inventory_cache")
REPORT_ROOT = Path("IMPORTANT_Photos_Library_Critical_Operation_Reports")
CONFIG_DIR = Path("data/local_config")
LIBRARY_HISTORY_PATH = CONFIG_DIR / "duplicate_cleanup_library_history.json"

library_history = load_json_file(LIBRARY_HISTORY_PATH, default={}) or {}

last_library_path = library_history.get("last_library_path")

if last_library_path:
    initial_dir = Path(last_library_path).parent
else:
    initial_dir = Path("/Volumes")

LIBRARY_PATH = Path(
    choose_photos_library_path(
        initial_dir=initial_dir,
    )
)


def make_safe_label_from_path(path):
    # Create a readable filesystem-safe label from the selected Photos Library name.
    name = Path(path).name

    if name.endswith(".photoslibrary"):
        name = name[:-len(".photoslibrary")]

    name = name.strip()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^\w.\-一-龥ぁ-んァ-ンー（）()]+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")

    if not name:
        name = "selected_photos_library"

    return name


LIBRARY_LABEL = make_safe_label_from_path(LIBRARY_PATH)
CACHE_NAME = LIBRARY_LABEL

library_history["last_library_path"] = str(LIBRARY_PATH)
library_history["last_library_label"] = LIBRARY_LABEL
library_history["last_selected_at"] = datetime.now().isoformat()
save_json_file(LIBRARY_HISTORY_PATH, library_history)

INVENTORY_CACHE_PATH = CACHE_DIR / f"{CACHE_NAME}.inventory.pkl.gz"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
REPORT_DIR = REPORT_ROOT / f"{RUN_TIMESTAMP}__Photos_Library_Duplicate_Cleanup__{LIBRARY_LABEL}"

print("LIBRARY_PATH:", LIBRARY_PATH)
print("LIBRARY_LABEL:", LIBRARY_LABEL)
print("CACHE_NAME:", CACHE_NAME)
print("INVENTORY_CACHE_PATH:", INVENTORY_CACHE_PATH)
print("REPORT_DIR:", REPORT_DIR)
print("LIBRARY_HISTORY_PATH:", LIBRARY_HISTORY_PATH)

LIBRARY_PATH: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
LIBRARY_LABEL: Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604
CACHE_NAME: Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604
INVENTORY_CACHE_PATH: data/inventory_cache/Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604.inventory.pkl.gz
REPORT_DIR: IMPORTANT_Photos_Library_Critical_Operation_Reports/20260610-100230__Photos_Library_Duplicate_Cleanup__Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604
LIBRARY_HISTORY_PATH: data/local_config/duplicate_cleanup_library_history.json


In [2]:
# ============================================================
# Cell 2. Load or build inventory
# ============================================================

import osxphotos

FORCE_REBUILD_INVENTORY = True

if INVENTORY_CACHE_PATH.exists() and not FORCE_REBUILD_INVENTORY:
    inventory = load_inventory_cache(
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )
    print("Loaded inventory cache:", INVENTORY_CACHE_PATH)
else:
    print("Building inventory from Photos Library:")
    print(LIBRARY_PATH)

    photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))
    osx_assets = photosdb.photos()

    inventory = build_inventory(osx_assets)

    save_inventory_cache(
        inventory=inventory,
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )

    print("Saved inventory cache:", INVENTORY_CACHE_PATH)

print_inventory_summary(inventory)

Building inventory from Photos Library:
/Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000
saved inventory cache: data/inventory_cache/Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604.inventory.pkl.gz
elapsed seconds: 1.02
Saved inventory cache: data/inventory_cache/Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604.inventory.pkl.gz
inventory assets: 71574
inventory albums: 5170
inventory folders: 35
movies: 6225
hidden: 0
favorites: 700
descriptions: 727
keywords: 23748


In [3]:
# ============================================================
# Cell 3. Fill identity fields
# ============================================================

fill_duplicate_cleanup_identity_fields(inventory)


Filled identity fields
asset count: 71574
base_id filled: 71574
unique_id filled: 71574
adjustment_signature filled: 22217
edited_duration_seconds filled: 1051

First 3 assets after fill:
--------------------------------------------------------------------------------
original_filename: IMG_0929.PNG
date: 2024-08-02T13:24:49+08:00
path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/9/99CE5BB6-01A6-4A44-BBFF-236A11762A01.png
path_edited: None
file_size_bytes: 369863
edited_duration_seconds: None
adjustment_signature: None
base_id: ('IMG_0929.PNG', '08-02 13:24:49.000000', 369863, None)
unique_id: (('IMG_0929.PNG', '08-02 13:24:49.000000', 369863, None), (('description', None), ('keywords', ()), ('favorite', False), ('hidden', False), ('album_titles', ('#台股 #沒睡飽不想看盤  2024年8月2日 開盤前大家就知道要大跌，我沒睡飽精神不好然後又跟小乖在聊

In [4]:
# ============================================================
# Cell 4. Group duplicate candidates
# ============================================================

unique_id_groups, assets_without_unique_id = group_assets_by_field(
    inventory,
    "photo_library_asset_unique_id",
)

duplicate_candidate_groups = {
    unique_id: group
    for unique_id, group in unique_id_groups.items()
    if len(group) > 1
}

print("assets:", len(inventory["assets"]))
print("generated unique_id count:", len(unique_id_groups))
print("assets without unique_id:", len(assets_without_unique_id))
print("duplicate candidate group count:", len(duplicate_candidate_groups))
print("duplicate candidate asset count:", sum(len(group) for group in duplicate_candidate_groups.values()))

if assets_without_unique_id:
    print()
    print("First assets without unique_id:")
    for asset in assets_without_unique_id[:10]:
        print(
            asset.get("original_filename"),
            asset.get("uuid"),
            asset.get("asset_scope"),
            asset.get("path"),
        )


assets: 71574
generated unique_id count: 71574
assets without unique_id: 0
duplicate candidate group count: 0
duplicate candidate asset count: 0


In [5]:
# ============================================================
# Cell 5. Analyze duplicate candidates
# ============================================================

duplicate_analysis = analyze_duplicate_candidate_groups(duplicate_candidate_groups)


analysis group count: 0
elapsed seconds: 0.0


In [6]:
# ============================================================
# Cell 6. Summary
# ============================================================

status_counts = count_records_by_status(duplicate_analysis)

delete_candidate_count = sum(
    len(record.get("delete_candidates") or [])
    for record in duplicate_analysis
)

print("status counts:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print("delete candidate asset count:", delete_candidate_count)

print()
print("First deletable duplicate groups:")
printed = 0

for record in duplicate_analysis:
    if record.get("status") != "DELETABLE_DUPLICATE":
        continue

    print("-" * 80)
    print("reason:", record.get("reason"))
    print("asset_count:", record.get("asset_count"))
    print("keep_assets:", len(record.get("keep_assets") or []))
    print("delete_candidates:", len(record.get("delete_candidates") or []))

    for asset in (record.get("keep_assets") or []):
        print("  KEEP:", asset["original_filename"], asset["date_added"], asset["path"])

    for asset in (record.get("delete_candidates") or []):
        print("  DELETE:", asset["original_filename"], asset["date_added"], asset["path"])

    printed += 1

    if printed >= 10:
        print("... more groups not printed")
        break


status counts:
delete candidate asset count: 0

First deletable duplicate groups:


In [7]:
# ============================================================
# Cell 7. Write permanent operation report
# ============================================================

REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_result = write_operation_report(
    report_dir=REPORT_DIR,
    duplicate_analysis=duplicate_analysis,
    inventory=inventory,
    assets_without_unique_id=assets_without_unique_id,
    duplicate_candidate_groups=duplicate_candidate_groups,
    run_timestamp=RUN_TIMESTAMP,
    library_id=LIBRARY_LABEL,
    library_path=LIBRARY_PATH,
    inventory_cache_path=INVENTORY_CACHE_PATH,
)

delete_candidate_rows = report_result["delete_candidate_rows"]
keep_asset_rows = report_result["keep_asset_rows"]
duplicate_review_asset_rows = report_result["duplicate_review_asset_rows"]
location_conflict_rows = report_result["location_conflict_rows"]
live_photo_candidate_rows = report_result["live_photo_candidate_rows"]
assets_without_unique_id_rows = report_result["assets_without_unique_id_rows"]
status_counts = report_result["status_counts"]
safety_counts = report_result["safety_counts"]

print("Wrote report to:", REPORT_DIR)
print("delete_candidate_rows:", len(delete_candidate_rows))
print("keep_asset_rows:", len(keep_asset_rows))
print("duplicate_review_asset_rows:", len(duplicate_review_asset_rows))
print("location_conflict_rows:", len(location_conflict_rows))
print("live_photo_candidate_rows:", len(live_photo_candidate_rows))
print("assets_without_unique_id_rows:", len(assets_without_unique_id_rows))

Photos Library Duplicate Cleanup Report

run_timestamp: 20260610-100230
library_id: Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604
library_path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
inventory_cache_path: data/inventory_cache/Photos_Library-iCloud-20250317（iCloud_20250325崩潰前最後的備份）--opened_by_macOS_Sequoia_on_20260604.inventory.pkl.gz

asset_count: 71574
assets_without_unique_id: 0
duplicate_candidate_group_count: 0
duplicate_candidate_asset_count: 0

status_counts:
{}

safety_counts:
{
  "live_photo_candidate_group_count": 0,
  "location_conflict_group_count": 0,
  "unreadable_original_group_count": 0,
  "sha_error_group_count": 0
}

delete_candidate_asset_count: 0
keep_asset_count: 0
duplicate_review_asset_count: 0
location_conflict_row_count: 0
live_photo_ca

In [8]:
# ============================================================
# Cell 8. Optional: print manual deletion list
# ============================================================
#
# This notebook does NOT delete anything from Photos Library.
# It only produces a report and delete candidate list.
#
# For actual deletion, review delete_candidates.tsv first.

for row in delete_candidate_rows[:100]:
    print(
        row["original_filename"],
        row["date"],
        row["date_added"],
        row["path"],
    )

if len(delete_candidate_rows) > 100:
    print("... more delete candidates not printed")
